<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/08-sequence-models-state-space-models.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **序列模型与状态空间模型** {#sequence-models-state-space-models}

第 07 章用共享空间卷积核编码二维局部性。序列数据具有不同的几何：观测按顺序到达，序列长度变化，一个元素的含义可能依赖很久以前的历史。序列模型必须决定**保存什么状态、信息如何跨位置移动，以及哪些计算可以并行执行**。

本章始终使用同一条真实数据主线：[OpenSLR SLR1 YESNO corpus](https://openslr.org/1/)，官方 [torchaudio `YESNO` 数据集](https://docs.pytorch.org/audio/main/generated/torchaudio.datasets.YESNO.html)也提供了对应说明。它包含一名男性说话者的 60 段录音，采样率为 8 kHz。每个文件名给出由八个希伯来语 yes/no 单词组成的序列，例如 `1_0_0_0_0_0_1_1.wav`。OpenSLR 说明该数据没有正式许可证，但可以自由用于任何目的。小型压缩包保存在本地，因此所有示例都能离线且可复现地运行。

每条波形会转换为 log-magnitude 短时傅里叶变换（STFT）帧。若音频为 $x[n]$、窗函数为 $w[n]$、帧索引为 $t$、频率 bin 为 $k$，则

$$
X[t,k]=\sum_{n=0}^{N-1}x[n+tH]w[n]e^{-j2\pi kn/N},
\qquad
z_t[k]=\log(1+|X[t,k]|).
$$

$N$ 是 FFT 大小，$H$ 是 hop length。模型接收的是变长序列 $Z\in\mathbb{R}^{T\times F}$，而不是原始采样点。每个频率通道都只用训练集统计量归一化。主要的小型分类任务预测**第一个口语单词**；CTC 部分则使用全部八个标签。

### **序列数据与时间依赖** {#sequential-data-temporal-dependence}

序列 $x_{1:T}=(x_1,\ldots,x_T)$ 不是无序集合。即使数值完全相同，打乱帧顺序也会改变语音的音素演化。三类常见输出结构是：

- **sequence-to-label：**整段序列对应一个输出，例如意图或说话者身份；
- **带对齐的 sequence-to-sequence：**每帧或每个 token 对应一个输出，例如序列标注；
- **未知对齐的 sequence-to-sequence：**目标序列更短，可使用 attention 或 CTC。

变长输入需要 mask 或 packed representation。Padding 是实现值，不是真实观测。如果填充帧会更新隐藏状态、影响归一化或进入池化平均，模型学到的就是 batch 排版方式。对于样本 $b$ 的长度 $L_b$，二值掩码 $M_{b,t}=1[t<L_b]$ 用于区分有效位置。

时间依赖还涉及方向。流式识别器在时刻 $t$ 只能使用 $x_{1:t}$，离线转写模型可以同时使用过去和未来上下文。双向模型能改善离线准确率，却违反因果延迟约束。因此可用上下文本身就是任务定义的一部分，而不只是架构选项。

<details>
<summary><strong>PyTorch：加载并归一化共享 YESNO 序列</strong></summary>

```python
import io
import math
import random
import tarfile
import wave
from pathlib import Path

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import DataLoader, Dataset

torch.set_num_threads(1)


def seed_everything(seed=808):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def read_pcm_wave(payload):
    with wave.open(io.BytesIO(payload), "rb") as stream:
        assert stream.getnchannels() == 1 and stream.getsampwidth() == 2
        sample_rate = stream.getframerate()
        samples = np.frombuffer(stream.readframes(stream.getnframes()), dtype="<i2").copy()
    return torch.tensor(samples, dtype=torch.float32) / 32768.0, sample_rate


archive_candidates = [
    Path("assets/data/waves_yesno.tar.gz"),
    Path("ipynb/Deep-Learning/assets/data/waves_yesno.tar.gz"),
]
archive = next(path for path in archive_candidates if path.exists())
records = []
window = torch.hann_window(256)
with tarfile.open(archive, "r:gz") as bundle:
    members = sorted(
        (m for m in bundle.getmembers() if m.isfile() and m.name.endswith(".wav")),
        key=lambda member: member.name,
    )
    for member in members:
        waveform, sample_rate = read_pcm_wave(bundle.extractfile(member).read())
        spectrum = torch.stft(
            waveform, n_fft=256, hop_length=160, win_length=256,
            window=window, return_complex=True,
        ).abs().transpose(0, 1)
        features = torch.log1p(spectrum)
        labels = torch.tensor([int(value) for value in Path(member.name).stem.split("_")])
        records.append({"name": member.name, "features": features, "labels": labels})

all_idx = np.arange(len(records))
first_labels = np.array([int(record["labels"][0]) for record in records])
train_idx, holdout_idx = train_test_split(
    all_idx, test_size=0.30, random_state=808, stratify=first_labels
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=808, stratify=first_labels[holdout_idx]
)

# Fit feature normalization on training frames only.
training_frames = torch.cat([records[i]["features"] for i in train_idx], dim=0)
feature_mean = training_frames.mean(0)
feature_std = training_frames.std(0).clamp_min(1e-5)
for record in records:
    record["features"] = (record["features"] - feature_mean) / feature_std


class YesNoDataset(Dataset):
    def __init__(self, indices):
        self.indices = list(map(int, indices))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        return records[self.indices[index]]


def collate_yesno(batch):
    sequences = [item["features"] for item in batch]
    lengths = torch.tensor([len(sequence) for sequence in sequences])
    padded = pad_sequence(sequences, batch_first=True)
    labels = torch.stack([item["labels"] for item in batch])
    mask = torch.arange(padded.shape[1]).unsqueeze(0) < lengths.unsqueeze(1)
    return padded, lengths, mask, labels


def yesno_loader(indices, shuffle=False, seed=808, batch_size=8):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        YesNoDataset(indices), batch_size=batch_size, shuffle=shuffle,
        generator=generator, collate_fn=collate_yesno,
    )


sample_batch = next(iter(yesno_loader(train_idx)))
padded, lengths, valid_mask, word_labels = sample_batch
assert len(records) == 60 and sample_rate == 8000
assert padded.shape[0] == 8 and padded.shape[2] == 129
assert valid_mask.sum(1).equal(lengths)
assert word_labels.shape == (8, 8)
assert set(train_idx).isdisjoint(test_idx)
print({"recordings": len(records), "split": (len(train_idx), len(val_idx), len(test_idx)),
       "batch shape": tuple(padded.shape), "length range": (int(lengths.min()), int(lengths.max()))})
```

</details>

下面所有章节复用该划分和归一化特征。由于数据只有一名说话者和 60 条语音，结果用于展示机制，而不能代表跨说话者的语音识别泛化。关于口音、语言或噪声条件的结论需要更大且按实体分组的数据集。

### **循环神经网络** {#recurrent-neural-networks}

循环神经网络把前缀 $x_{1:t}$ 压缩到隐藏状态 $h_t$：

$$
a_t=W_{xh}x_t+W_{hh}h_{t-1}+b_h,
\qquad
h_t=\phi(a_t),
\qquad
o_t=W_{hy}h_t+b_y.
$$

$x_t\in\mathbb{R}^{F}$ 是一帧声学特征，$h_t\in\mathbb{R}^{H}$ 是学习到的历史摘要，同一组矩阵在每个时间步复用。权重共享使模型可以处理任意长度，并表达时间平稳的转移规则；同时也产生顺序依赖：在计算 $h_{t-1}$ 前无法得到 $h_t$。

对序列分类，应使用最后一个**有效**隐藏状态，而不是最终 padding 位置。`pack_padded_sequence` 能让 PyTorch 跳过填充更新；也可以在未打包张量中通过正确 mask 提取 `output[b, L_b-1]`。对逐帧任务则保留每个有效输出，并在损失中屏蔽 padding。

隐藏状态是信息瓶颈：它必须持续吸收新证据，同时保存未来预测需要的信息。普通 `tanh` RNN 能高效处理短依赖，但反复的 Jacobian 乘积使长记忆难以优化。门控单元会改变这条状态更新路径。

![RNN 单元沿时间展开后，可以看到共享参数以及连续输入之间的隐藏状态依赖。](assets/dl08-unfolded-rnn.svg){fig-align="center" width="74%" fig-alt="沿时间展开的循环神经网络，共享单元参数并传递隐藏状态。"}

*图片来源：[Dive into Deep Learning, Recurrent Neural Networks](https://d2l.ai/chapter_recurrent-neural-networks/rnn.html)，CC BY-SA 4.0。*

<details>
<summary><strong>PyTorch：在 YESNO 第一个单词任务上训练普通 RNN</strong></summary>

```python
class RecurrentFirstWord(nn.Module):
    def __init__(self, cell="rnn", input_size=129, hidden_size=32, bidirectional=False):
        super().__init__()
        cells = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}
        self.encoder = cells[cell](
            input_size, hidden_size, batch_first=True,
            nonlinearity="tanh" if cell == "rnn" else None,
            bidirectional=bidirectional,
        ) if cell == "rnn" else cells[cell](input_size, hidden_size, batch_first=True,
                                             bidirectional=bidirectional)
        directions = 2 if bidirectional else 1
        self.head = nn.Linear(hidden_size * directions, 2)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, state = self.encoder(packed)
        hidden = state[0] if isinstance(state, tuple) else state
        if self.encoder.bidirectional:
            representation = torch.cat([hidden[-2], hidden[-1]], dim=-1)
        else:
            representation = hidden[-1]
        return self.head(representation)


@torch.no_grad()
def sequence_accuracy(model, indices):
    model.eval()
    correct = total = 0
    for xb, lens, _, labels in yesno_loader(indices, batch_size=8):
        prediction = model(xb, lens).argmax(1)
        correct += int((prediction == labels[:, 0]).sum())
        total += len(labels)
    return correct / total


def fit_sequence_classifier(model, epochs=12, seed=809):
    seed_everything(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
    for _ in range(epochs):
        model.train()
        for xb, lens, _, labels in yesno_loader(train_idx, shuffle=True, seed=seed):
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(xb, lens), labels[:, 0])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
    return {"validation": sequence_accuracy(model, val_idx),
            "test": sequence_accuracy(model, test_idx)}


seed_everything(809)
vanilla_rnn = RecurrentFirstWord("rnn")
rnn_result = fit_sequence_classifier(vanilla_rnn, epochs=10, seed=809)
assert vanilla_rnn(padded, lengths).shape == (8, 2)
assert all(0 <= value <= 1 for value in rnn_result.values())
print(rnn_result)
```

</details>

这里分数的重要性低于状态接口。验证和测试各只有九条、且来自同一个说话者，一次错误就会让准确率变化约 11 个百分点。正确结论是实现能够处理真实变长音频，而不是一个小 RNN 已经建立语音基准。

### **随时间反向传播** {#backpropagation-through-time}

随时间反向传播（BPTT）把循环计算展开为深度为 $T$ 的计算图，再应用反向模式自动微分。若总损失 $L=\sum_t\ell_t$，共享循环矩阵会收到每次使用产生的梯度贡献：

$$
\frac{\partial L}{\partial W_{hh}}
=\sum_{t=1}^{T}
\frac{\partial L}{\partial h_t}
\frac{\partial h_t}{\partial W_{hh}}.
$$

从时刻 $t$ 回溯到时刻 $k$ 的影响包含 Jacobian 乘积：

$$
\frac{\partial h_t}{\partial h_k}
=\prod_{j=k+1}^{t}
\frac{\partial h_j}{\partial h_{j-1}}.
$$

若典型奇异值小于一，梯度会指数消失；大于一则可能爆炸。`tanh` 与 sigmoid 在饱和区域的导数会进一步收缩路径。梯度裁剪只能限制已经发生的爆炸，不能恢复已经消失的信息。

完整 BPTT 保存所有时间步的激活，内存随序列长度增长。**截断 BPTT**分窗口处理序列，把隐藏状态数值传到下一窗口，但在窗口边界将其 detach。这样可以减少内存和更新延迟，却会把跨边界依赖视为常量。因此截断长度是建模近似，而不只是系统参数。

<details>
<summary><strong>PyTorch：在一条真实声学序列上追踪梯度衰减</strong></summary>

```python
# Use the first 50 normalized acoustic frames and retain each hidden state gradient.
acoustic_prefix = records[int(train_idx[0])]["features"][:50]
input_projection = nn.Linear(129, 24, bias=False)
recurrent = nn.Linear(24, 24, bias=False)
with torch.no_grad():
    recurrent.weight.mul_(0.35)  # Create a contractive transition for a visible diagnostic.

hidden = torch.zeros(24)
hidden_states = []
for frame in acoustic_prefix:
    hidden = torch.tanh(input_projection(frame) + recurrent(hidden))
    hidden.retain_grad()
    hidden_states.append(hidden)

loss = hidden_states[-1].pow(2).mean()
loss.backward()
gradient_norms = torch.tensor([state.grad.norm().item() for state in hidden_states])

assert len(gradient_norms) == 50
assert torch.isfinite(gradient_norms).all()
assert gradient_norms[-1] > gradient_norms[0]
print({"first-step gradient": gradient_norms[0].item(),
       "last-step gradient": gradient_norms[-1].item(),
       "ratio": (gradient_norms[-1] / gradient_norms[0].clamp_min(1e-30)).item()})
```

</details>

代码有意使用收缩矩阵，让一种失败模式清晰可见；训练后的网络可能在不同方向同时出现增长与衰减。应按层和时间诊断梯度，并区分优化失败与任务本身并不需要远距离上下文。

### **长短期记忆网络** {#long-short-term-memory}

LSTM 引入具有加性更新的 cell state $c_t$：

$$
\begin{aligned}
i_t&=\sigma(W_i x_t+U_i h_{t-1}+b_i),\\
f_t&=\sigma(W_f x_t+U_f h_{t-1}+b_f),\\
g_t&=\tanh(W_g x_t+U_g h_{t-1}+b_g),\\
o_t&=\sigma(W_o x_t+U_o h_{t-1}+b_o),\\
c_t&=f_t\odot c_{t-1}+i_t\odot g_t,\\
h_t&=o_t\odot\tanh(c_t).
\end{aligned}
$$

遗忘门 $f_t$ 保留或移除旧记忆，输入门 $i_t$ 控制新内容，候选 $g_t$ 提议新内容，输出门 $o_t$ 暴露选中的记忆。直接导数 $\partial c_t/\partial c_{t-1}=f_t$ 提供可调节的梯度通道。当 $f_t$ 保持接近一时，记忆可以比反复通过无约束 `tanh` 转移保存得更久。

门控并不能消除所有长距离问题。饱和门学习缓慢，cell 值可能漂移，每步还需要四组仿射投影。长序列在训练和推理中依然是顺序执行。当数据规模适中、流式状态重要、固定记忆比全对交互更合适时，LSTM 很有吸引力。

![LSTM 利用输入门、遗忘门和输出门，把持久 cell memory 与对外隐藏状态分离。](assets/dl08-lstm-memory.svg){fig-align="center" width="76%" fig-alt="包含 cell state、hidden state 和三个门的 LSTM 单元图。"}

*图片来源：依据标准 LSTM 公式与 [Hochreiter and Schmidhuber (1997)](https://www.bioinf.jku.at/publications/older/2604.pdf) 绘制的本地教学图。*

<details>
<summary><strong>PyTorch：在同一 YESNO 协议下训练 LSTM</strong></summary>

```python
seed_everything(810)
lstm_classifier = RecurrentFirstWord("lstm")
lstm_result = fit_sequence_classifier(lstm_classifier, epochs=12, seed=810)

parameter_count = lambda model: sum(parameter.numel() for parameter in model.parameters())
expected_lstm_core = 4 * 32 * (129 + 32 + 2)  # weights plus two bias vectors per gate in PyTorch
actual_lstm_core = sum(p.numel() for p in lstm_classifier.encoder.parameters())

assert actual_lstm_core == expected_lstm_core
assert lstm_classifier(padded, lengths).shape == (8, 2)
print({"core parameters": actual_lstm_core, "validation/test": lstm_result})
```

</details>

四组门对应的参数量也是权衡的一部分。模型比较既可以匹配隐藏宽度，也可以匹配参数预算；这是两个不同实验，可能得出不同结论。

### **门控循环单元** {#gated-recurrent-units}

GRU 合并 cell state 与 hidden state，通常使用三组仿射门：

$$
z_t=\sigma(W_zx_t+U_zh_{t-1}),
\qquad
r_t=\sigma(W_rx_t+U_rh_{t-1}),
$$

$$
\widetilde h_t=\tanh(W_hx_t+U_h(r_t\odot h_{t-1})),
\qquad
h_t=(1-z_t)\odot\widetilde h_t+z_t\odot h_{t-1}.
$$

更新门 $z_t$ 在保留旧状态与接受候选之间插值；重置门 $r_t$ 控制旧状态对候选的贡献。有些资料会对调 $z_t$ 两侧的系数，因此理解 GRU 时应查看公式或实现，而不是只凭门的名称。

在相同隐藏宽度下，GRU 通常比 LSTM 参数更少、每步计算更低。LSTM 提供独立控制的 cell state，GRU 提供更简单的状态接口。二者都不是所有任务上的绝对优胜者；序列长度、数据规模、延迟和超参数预算往往比单元名称更重要。

<details>
<summary><strong>PyTorch：在相同隐藏宽度下比较 GRU 与 LSTM</strong></summary>

```python
seed_everything(811)
gru_classifier = RecurrentFirstWord("gru")
gru_result = fit_sequence_classifier(gru_classifier, epochs=12, seed=811)

gru_core = sum(p.numel() for p in gru_classifier.encoder.parameters())
lstm_core = sum(p.numel() for p in lstm_classifier.encoder.parameters())
comparison_batch = padded[:4]
comparison_lengths = lengths[:4]

assert gru_classifier(comparison_batch, comparison_lengths).shape == (4, 2)
assert gru_core < lstm_core
print({"GRU core parameters": gru_core, "LSTM core parameters": lstm_core,
       "GRU validation/test": gru_result})
```

</details>

由于运行使用不同初始化且验证集极小，分数不能建立单元排名。可靠观察是架构性的：GRU 使用三组门和一个循环状态，LSTM 使用四组门并分离 cell state 与 hidden state。

### **双向与编码器-解码器循环模型** {#bidirectional-encoder-decoder-recurrent-models}

双向循环模型计算

$$
\overrightarrow h_t=F(x_t,\overrightarrow h_{t-1}),
\qquad
\overleftarrow h_t=B(x_t,\overleftarrow h_{t+1}),
\qquad
h_t=[\overrightarrow h_t;\overleftarrow h_t].
$$

每个位置都获得过去和未来上下文。这适用于离线标注与转写，但只有等整段语音结束后才能得到最终反向状态。分块双向系统用有限未来上下文换取有界延迟；严格流式系统必须保持因果。

编码器-解码器把输入表示与输出生成分开。编码器将声学帧映射到隐藏状态，解码器自回归预测输出 token：

$$
p(y_{1:U}\mid x_{1:T})=\prod_{u=1}^{U}p(y_u\mid y_{<u},x_{1:T}).
$$

固定的最终编码器状态会形成瓶颈。Attention 让每个解码步从全部编码器状态中提取不同的加权组合；CTC 则在单调条件独立假设下去掉自回归解码器。Teacher forcing 在训练时提供真实前一 token，推理时却使用模型自己的输出，因此产生 exposure bias。

<details>
<summary><strong>PyTorch：暴露双向状态并执行一次 teacher-forced 解码</strong></summary>

```python
bi_encoder = nn.GRU(129, 24, batch_first=True, bidirectional=True)
packed = pack_padded_sequence(padded, lengths.cpu(), batch_first=True, enforce_sorted=False)
packed_output, bi_state = bi_encoder(packed)
encoder_memory, _ = torch.nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)

token_embedding = nn.Embedding(3, 16)  # 0/1 words plus start token 2
decoder = nn.GRU(input_size=16 + 48, hidden_size=48, batch_first=True)
output_head = nn.Linear(48, 2)

start = torch.full((padded.shape[0], 1), 2, dtype=torch.long)
teacher_inputs = torch.cat([start, word_labels[:, :-1]], dim=1)
context = torch.cat([bi_state[-2], bi_state[-1]], dim=-1).unsqueeze(1).expand(-1, 8, -1)
decoder_input = torch.cat([token_embedding(teacher_inputs), context], dim=-1)
decoder_output, _ = decoder(decoder_input)
teacher_forced_logits = output_head(decoder_output)

assert encoder_memory.shape[:2] == padded.shape[:2]
assert encoder_memory.shape[2] == 48
assert teacher_forced_logits.shape == (8, 8, 2)
print({"encoder memory": tuple(encoder_memory.shape),
       "decoder logits": tuple(teacher_forced_logits.shape)})
```

</details>

示例重复使用最终上下文，以明确展示固定瓶颈。Attention decoder 会在每个输出步计算不同上下文。不论采用哪种方式，数据访问契约决定双向编码是否允许。

### **时间卷积与连接时序分类** {#temporal-convolution-ctc}

时间卷积网络（TCN）沿时间轴使用一维卷积。空洞因果层可以指数扩大上下文：对核大小 $K$ 和 dilation $1,2,4,\ldots,2^{L-1}$，感受野为

$$
R=1+(K-1)\sum_{l=0}^{L-1}2^l
=1+(K-1)(2^L-1).
$$

不同于 recurrence，同一个卷积层中的所有位置都能并行计算。因果 TCN 只在过去方向 padding；对称 TCN 使用未来上下文。除非增加深度或 dilation，否则它的记忆是有限的；不合适的 dilation 模式还会遗漏局部交互。

当输入有 $T$ 帧、目标有 $U\le T$ 个符号且帧级对齐未知时，可以使用 Connectionist Temporal Classification（CTC）。加入 blank 符号 $\varnothing$。路径 $\pi\in(\mathcal{V}\cup\{\varnothing\})^T$ 通过移除 blank 和相邻重复符号得到目标。序列概率对全部有效路径求和：

$$
p(y\mid x)=\sum_{\pi:\mathcal{B}(\pi)=y}\prod_{t=1}^{T}p(\pi_t\mid x).
$$

动态规划无需枚举路径即可计算该和。CTC 假设给定编码器后各帧预测条件独立，并强制单调顺序。它不能直接表达任意输出重排，重复标签之间还必须由 blank 分隔。

![CTC 把包含 blank 与重复的多条帧级路径折叠为同一个较短标签序列。](assets/dl08-ctc-alignment.svg){fig-align="center" width="76%" fig-alt="CTC 对齐图，展示帧级路径如何折叠成较短 yes-no 标签序列。"}

*图片来源：依据 [Graves et al. (2006)](https://www.cs.toronto.edu/~graves/icml_2006.pdf) 中的 CTC collapse operator 绘制的本地教学图。*

<details>
<summary><strong>PyTorch：在 YESNO 上连接 TCN 编码器与 CTC 目标</strong></summary>

```python
class TemporalBlock(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()
        self.dilation = dilation
        self.conv = nn.Conv1d(channels, channels, 3, dilation=dilation)

    def forward(self, x):
        # Left-only padding makes the output causal and keeps its length.
        x = F.pad(x, (2 * self.dilation, 0))
        return F.relu(self.conv(x))


projected = nn.Conv1d(129, 32, 1)(padded.transpose(1, 2))
tcn = nn.Sequential(TemporalBlock(32, 1), TemporalBlock(32, 2), TemporalBlock(32, 4))
tcn_features = tcn(projected).transpose(1, 2)
ctc_head = nn.Linear(32, 3)  # labels 0/1 and blank index 2
log_probabilities = ctc_head(tcn_features).log_softmax(-1).transpose(0, 1)

flat_targets = word_labels.flatten()
target_lengths = torch.full((len(word_labels),), 8, dtype=torch.long)
ctc_loss = F.ctc_loss(
    log_probabilities, flat_targets, lengths, target_lengths,
    blank=2, zero_infinity=True,
)

assert tcn_features.shape[:2] == padded.shape[:2]
assert log_probabilities.shape == (padded.shape[1], padded.shape[0], 3)
assert torch.isfinite(ctc_loss)
assert 1 + (3 - 1) * (1 + 2 + 4) == 15
print({"TCN receptive field": 15, "CTC loss": ctc_loss.item()})
```

</details>

未训练损失只验证对齐管线。完整系统还需要训练编码器，用 greedy collapse 或 beam search 解码，并在未参与拟合的说话者上报告序列错误率。

### **长距离依赖与并行性限制** {#long-range-dependency-parallelism-limits}

序列架构在状态大小、依赖路径长度、训练并行性和推理内存之间进行权衡。

- RNN 中，位置 1 的信息要经过 $T-1$ 次转移才能到达位置 $T$。时间维训练是顺序的，但流式推理只保存固定大小状态。
- 空洞 TCN 在覆盖上下文内可获得对数级路径长度，并在每层内部并行训练；流式推理需要缓存各层近期激活。
- 完整 self-attention 在一层内连接任意两个位置，但 score matrix 使用 $O(T^2)$ 内存/计算。自回归推理的 key-value cache 会随上下文增长。
- 状态空间模型的循环 scan 可用固定状态流式运行，而时间不变形式在训练中可作为并行卷积计算。

这些渐近结论不能直接预测墙钟速度。Kernel fusion、序列长度、隐藏宽度、batch size、内存带宽和加速器利用率都很重要。YESNO 的 STFT 序列只有数百帧；在长音频、基因组或高频传感器数据中，二次 attention 才会更突出。

建模问题同样重要。名义上很长的上下文不能证明模型真正保留或使用远距离信息。扰动测试、梯度归因、受控检索任务以及性能随上下文截断的变化，才能揭示功能性上下文。

<details>
<summary><strong>Python：用观测长度比较依赖与内存缩放</strong></summary>

```python
observed_length = int(lengths.max())
hidden_width = 64
layers = 8
kernel_size = 3
tcn_receptive_field = 1 + (kernel_size - 1) * (2 ** layers - 1)

scaling = {
    "RNN transition path": observed_length - 1,
    "dilated TCN receptive field": tcn_receptive_field,
    "attention score elements per head": observed_length ** 2,
    "streaming recurrent state elements": hidden_width,
    "autoregressive KV elements per layer (one K and one V)": 2 * observed_length * hidden_width,
}

assert tcn_receptive_field >= observed_length
assert scaling["attention score elements per head"] > scaling["RNN transition path"]
print(scaling)
```

</details>

这里使用真实 batch 长度，但结果仍是资源模型而非性能基准。正式基准需要预热、同步计时、目标精度和实际硬件上的峰值内存。

### **结构化状态空间模型与 S4** {#structured-state-space-models-s4}

连续时间线性状态空间模型（SSM）为

$$
\frac{d h(t)}{dt}=Ah(t)+Bx(t),
\qquad
y(t)=Ch(t)+Dx(t),
$$

其中状态 $h(t)\in\mathbb{R}^{N}$ 压缩历史。用步长 $\Delta$ 离散化后得到

$$
h_k=\overline A h_{k-1}+\overline Bx_k,
\qquad
y_k=Ch_k+Dx_k.
$$

零阶保持下，若 $A$ 可逆，则 $\overline A=e^{\Delta A}$、$\overline B=A^{-1}(e^{\Delta A}-I)B$。更简单的 Euler 近似使用 $\overline A=I+\Delta A$、$\overline B=\Delta B$，但步长过大时可能不稳定。

当 $A,B,C$ 时间不变时，展开可得到卷积：

$$
y_k=\sum_{j=0}^{k}K_{k-j}x_j,
\qquad
K_i=C\overline A^i\overline B,
$$

另加直接项 $Dx_k$。这种**循环-卷积对偶**允许用固定状态进行流式 recurrence，同时用并行卷积处理完整序列。

S4 通过对 $A$ 施加结构化参数化并利用高效核生成，让较大的状态维度变得可计算。其 HiPPO 风格初始化旨在保存连续历史的信息，而不是采用任意转移。并非每个 SSM 都是 S4；S4 结合了状态空间视角、特定长记忆初始化和结构化数值算法。

![线性时不变状态空间 recurrence 可以展开为一维卷积核。](assets/dl08-ssm-duality.svg){fig-align="center" width="76%" fig-alt="对比状态空间循环扫描与等价卷积核的示意图。"}

*图片来源：依据 [Gu, Goel, and Ré, Efficiently Modeling Long Sequences with Structured State Spaces](https://arxiv.org/abs/2111.00396) 绘制的本地教学图。*

<details>
<summary><strong>PyTorch：在 YESNO 能量序列上验证循环-卷积对偶</strong></summary>

```python
# Collapse the first utterance's spectrum to one scalar energy signal per frame.
u = records[int(train_idx[0])]["features"][:80].pow(2).mean(-1)
state_size = 12
continuous_A = -torch.exp(torch.linspace(-2.0, 0.5, state_size))
delta = 0.05
A_bar = torch.exp(delta * continuous_A)
B_bar = torch.randn(state_size) * 0.05
C = torch.randn(state_size)

state = torch.zeros(state_size)
recurrent_output = []
for value in u:
    state = A_bar * state + B_bar * value
    recurrent_output.append(C @ state)
recurrent_output = torch.stack(recurrent_output)

kernel = torch.stack([C @ (A_bar.pow(lag) * B_bar) for lag in range(len(u))])
convolution_output = torch.stack([
    (kernel[:time + 1] * u[:time + 1].flip(0)).sum()
    for time in range(len(u))
])

assert torch.allclose(recurrent_output, convolution_output, atol=1e-5)
assert A_bar.abs().max() < 1  # Stable decaying modes for this diagonal example.
print({"sequence length": len(u), "state size": state_size,
       "maximum duality error": (recurrent_output - convolution_output).abs().max().item()})
```

</details>

对角模型揭示了代数关系，但省略 S4 的结构化矩阵机制与快速核算法。这是组件级机制实现：足以验证双重视角，但不会假装复现优化后的 S4 库。

### **Mamba 与选择性状态空间模型** {#mamba-selective-state-space-models}

线性时不变 SSM 在所有位置使用相同动力学，效率很高，却限制了**基于内容的推理**：模型无法直接决定某个输入应被保存而另一个应被忽略。Mamba 让关键 SSM 参数依赖输入。在简化的选择性更新中，

$$
\Delta_t=\operatorname{softplus}(W_\Delta x_t),
\quad B_t=W_Bx_t,
\quad C_t=W_Cx_t,
$$

$$
h_t=e^{\Delta_t A}\odot h_{t-1}+\Delta_t B_t\odot u_t,
\qquad
y_t=C_t^\top h_t+D u_t.
$$

$A$ 仍是学习得到的稳定基础动力学，而 $\Delta_t$、$B_t$、$C_t$ 让内容控制时间尺度、写入和读取。较强衰减可以重置记忆，慢模态可以保存信息，输入相关读取向量则暴露不同状态分量。

选择性打破了线性时不变 SSM 的固定卷积核。Mamba 通过硬件友好的并行 selective scan 恢复效率，并在周围模块中加入局部卷积和门控投影。其 scan 对序列长度是线性的，但实际速度仍依赖融合内核、状态宽度、精度和内存流量。

Mamba 并不只是“线性复杂度的 attention”。Attention 显式比较 token 对，可以直接检索某个早期表示；选择性 SSM 把历史压缩进有限状态。这对长流式数据很有价值，但若状态和动力学不支持精确关联检索，它就不如显式 attention 直接。

![选择性状态空间动力学允许输入内容在每一步控制信息的写入、保留与读取强度。](assets/dl08-selective-ssm.svg){fig-align="center" width="76%" fig-alt="输入相关步长、写入和读取参数组成的选择性状态空间模型图。"}

*图片来源：依据 [Gu and Dao, Mamba: Linear-Time Sequence Modeling with Selective State Spaces](https://arxiv.org/abs/2312.00752) 绘制的本地教学图。*

<details>
<summary><strong>PyTorch：在声学帧上运行简化 selective scan</strong></summary>

```python
class SelectiveAcousticSSM(nn.Module):
    def __init__(self, input_size=129, state_size=16):
        super().__init__()
        self.input_projection = nn.Linear(input_size, 1)
        self.delta_projection = nn.Linear(input_size, state_size)
        self.write_projection = nn.Linear(input_size, state_size)
        self.read_projection = nn.Linear(input_size, state_size)
        self.log_decay = nn.Parameter(torch.linspace(-2.0, 0.0, state_size))
        self.direct = nn.Parameter(torch.tensor(0.0))

    def forward(self, x, mask):
        batch, time, _ = x.shape
        state = x.new_zeros(batch, self.log_decay.numel())
        outputs = []
        A = -torch.exp(self.log_decay)
        for step in range(time):
            frame = x[:, step]
            u = self.input_projection(frame)
            delta = F.softplus(self.delta_projection(frame))
            write = torch.tanh(self.write_projection(frame))
            read = torch.tanh(self.read_projection(frame))
            candidate = torch.exp(delta * A) * state + delta * write * u
            active = mask[:, step].unsqueeze(1)
            state = torch.where(active, candidate, state)
            outputs.append((read * state).sum(-1, keepdim=True) + self.direct * u)
        return torch.stack(outputs, dim=1)


selective_ssm = SelectiveAcousticSSM()
selective_output = selective_ssm(padded[:4], valid_mask[:4])

# Padded steps leave state unchanged internally; outputs retain a regular tensor shape.
assert selective_output.shape == (4, padded.shape[1], 1)
assert torch.isfinite(selective_output).all()
loss = selective_output[valid_mask[:4]].pow(2).mean()
loss.backward()
assert selective_ssm.log_decay.grad is not None
print({"output": tuple(selective_output.shape), "loss": loss.item()})
```

</details>

Python 循环刻意保持透明，因此速度很慢。生产 Mamba block 还会使用扩展通道、局部因果卷积、门控、归一化、残差连接和融合 scan。性能敏感场景应使用维护良好的实现；显式循环用于理解状态语义与 mask。

### **Attention-SSM 混合架构** {#attention-ssm-hybrid-architectures}

Attention 与 SSM 提供互补的通信机制：

- attention 提供内容寻址的成对交互和直接检索；
- SSM 提供线性时间扫描、压缩循环状态和高效流式处理；
- 局部卷积提供短距离模式提取和稳定高吞吐内核。

混合模型可以交替使用不同模块，在 SSM 层之间放置少量 attention 层，或把通道分为并行 attention 与 state-space 分支。目的不是装饰架构，而是用稀疏 attention 恢复精确检索或全局协调，同时让大多数层保持线性序列缩放。

混合也引入接口问题：分支输出要有兼容维度和归一化；causal mask 必须一致。如果 attention 分支没有屏蔽 padding，而 SSM 分支会冻结填充状态，融合结果就会泄漏 batch 排版。流式部署还必须同时缓存循环 SSM 状态和保留窗口的 attention key/value。

计算轮廓取决于 attention 出现频率。如果每个 block 都含全局 attention，长序列上仍由二次成本主导；若每 $m$ 层才有 attention，或只使用局部窗口，平衡会不同。应报告实际层模式和上下文窗口，而不能根据单个组件把整个模型称为“线性”。

<details>
<summary><strong>PyTorch：在同一音频上融合 masked attention 与 selective state</strong></summary>

```python
class HybridSequenceBlock(nn.Module):
    def __init__(self, input_size=129, model_size=32):
        super().__init__()
        self.project = nn.Linear(input_size, model_size)
        self.attention = nn.MultiheadAttention(model_size // 2, num_heads=4, batch_first=True)
        self.ssm = SelectiveAcousticSSM(input_size=model_size // 2, state_size=12)
        self.fuse = nn.Linear(model_size, model_size)
        self.norm = nn.LayerNorm(model_size)

    def forward(self, x, mask):
        residual = self.project(x)
        attention_input, state_input = residual.chunk(2, dim=-1)
        attention_output, _ = self.attention(
            attention_input, attention_input, attention_input,
            key_padding_mask=~mask, need_weights=False,
        )
        state_scalar = self.ssm(state_input, mask)
        state_output = state_scalar.expand(-1, -1, state_input.shape[-1])
        mixed = self.fuse(torch.cat([attention_output, state_output], dim=-1))
        return self.norm(residual + mixed).masked_fill(~mask.unsqueeze(-1), 0.0)


hybrid = HybridSequenceBlock()
hybrid_output = hybrid(padded[:4], valid_mask[:4])
assert hybrid_output.shape == (4, padded.shape[1], 32)
assert torch.all(hybrid_output[~valid_mask[:4]] == 0)
print({"hybrid output": tuple(hybrid_output.shape),
       "valid vectors": int(valid_mask[:4].sum())})
```

</details>

这里的分支扩展是教学实现，不代表某个已发表的混合配方。它让关键契约可以测试：两个分支处理相同有效时间步，融合保持形状，并明确移除 padding 输出。

### **RNN、Transformer 与 SSM 对比** {#rnns-transformers-ssms-compared}

不存在脱离上下文长度、延迟、数据规模、硬件和任务依赖类型的最佳序列家族。

| 家族 | 训练交互方式 | 流式状态/缓存 | 长距离优势 | 主要限制 |
|---|---|---|---|---|
| 普通 RNN | 顺序 recurrence | 固定 hidden state | 紧凑但难优化 | 梯度消失/爆炸与短瓶颈 |
| LSTM/GRU | 门控顺序 recurrence | 固定循环状态 | 更强的学习型记忆 | 并行性有限且每步有门控成本 |
| TCN | 并行局部/空洞卷积 | 有限激活缓存 | 可控感受野 | 有限上下文与 dilation 缺口 |
| 完整 Transformer | 并行全对 attention | 随上下文增长的 KV cache | 直接内容检索 | 二次训练交互与缓存增长 |
| 结构化 LTI SSM/S4 | 并行卷积或循环 scan | 固定状态 | 高效长滤波 | 基础动力学不依赖内容 |
| 选择性 SSM/Mamba | 硬件友好 selective scan | 固定选择性状态 | 内容控制的压缩记忆 | 无显式成对检索且依赖优化内核 |
| 混合架构 | 选定组合 | 状态与部分 KV cache | 平衡检索与线性扫描 | 设计和实现更复杂 |

在 YESNO 数据上，所有方法接收相同的归一化 log-STFT 帧与 mask。这种一致性揭示真正变化的对象：状态转移、依赖路径、对齐目标或通信算子；同时也暴露数据无法回答的问题。一名说话者和 60 条录音不能证明跨说话者或跨语言鲁棒性。

<details>
<summary><strong>PyTorch：验证统一的序列模型接口</strong></summary>

```python
interface_batch = padded[:3]
interface_lengths = lengths[:3]
interface_mask = valid_mask[:3]

rnn_logits = vanilla_rnn(interface_batch, interface_lengths)
lstm_logits = lstm_classifier(interface_batch, interface_lengths)
gru_logits = gru_classifier(interface_batch, interface_lengths)
ssm_features = selective_ssm(interface_batch, interface_mask)
hybrid_features = hybrid(interface_batch, interface_mask)

outputs = {
    "RNN logits": tuple(rnn_logits.shape),
    "LSTM logits": tuple(lstm_logits.shape),
    "GRU logits": tuple(gru_logits.shape),
    "selective SSM sequence": tuple(ssm_features.shape),
    "hybrid sequence": tuple(hybrid_features.shape),
}
assert rnn_logits.shape == lstm_logits.shape == gru_logits.shape == (3, 2)
assert ssm_features.shape[:2] == hybrid_features.shape[:2] == interface_batch.shape[:2]
print(outputs)
```

</details>

公平的预测比较还需要加入等价输出头，在匹配预算下调优每个家族，重复多个随机种子，并报告质量、峰值内存、吞吐量与流式延迟。接口相同是比较的必要条件，但不会自动让容量和优化预算相同。

### **本章对比与总结** {#chapter-comparison-summary}

序列建模是在有序观测之间设计信息通路。核心问题不只是“使用哪一层”，还包括：哪些上下文合法可用、如何排除 padding、如何优化长依赖、输出怎样与输入对齐，以及训练和推理中的状态如何缩放。

| 机制 | 状态更新或交互 | 最适用场景 | 诊断重点 |
|---|---|---|---|
| 普通 recurrence | 共享非线性转移 | 短流式序列与概念基线 | 随时间梯度与最后有效状态 |
| LSTM | 门控加性 cell state | 中长流式依赖 | 门饱和、状态漂移与循环延迟 |
| GRU | 单状态中的门控插值 | 紧凑循环基线 | 匹配宽度与匹配预算的区别 |
| 双向编码器 | 过去与未来 recurrence | 离线标注/转写 | 未来上下文造成的延迟违规 |
| 编码器-解码器 | 条件自回归输出 | 输入输出长度不同 | 瓶颈、teacher forcing 与 exposure bias |
| TCN | 局部/空洞时间核 | 并行有限上下文处理 | 感受野覆盖与因果 padding |
| CTC | 对单调对齐路径求和 | 未分段语音/手写 | blank/repeat collapse 与长度约束 |
| S4 风格 SSM | 结构化线性状态与卷积对偶 | 长序列与流式处理 | 离散化稳定性与核生成 |
| Mamba 风格 SSM | 输入相关 selective scan | 长且内容敏感的数据流 | 状态容量、mask 与优化内核 |
| 混合架构 | attention 检索加压缩状态 | 同时需要精确检索和长上下文 | 分支 mask、缓存策略与真实复杂度 |

本章的连续实验路线如下：

1. 解析官方波形，从文件名提取标签，并只用训练帧拟合特征归一化。
2. 用显式有效 mask 对变长 batch 进行 padding。
3. 用循环分类器暴露状态语义，比较普通 RNN、LSTM 和 GRU。
4. 展开真实声学前缀，观察 BPTT 梯度行为。
5. 保留逐帧编码器输出，用于双向模型、解码、时间卷积和 CTC。
6. 验证时间不变 SSM 的循环输出与卷积输出完全一致。
7. 在透明 selective scan 中让状态转移依赖输入内容。
8. 只有在维度、mask 和因果性对齐后，才融合 attention 与选择性状态。

实践选择应有条件：固定状态流式处理和中等长度占主导时选择门控 recurrence；需要并行训练且有限感受野自然时选择时间卷积；直接内容检索重要时选择 attention；长序列线性扫描和紧凑状态重要时选择 SSM；任务确实同时需要检索与压缩记忆时才选择混合架构。最后应在代表性长度和硬件上评估，而不能从渐近符号直接推断系统行为。

第 09 章将在这些序列基础上深入研究 attention 与 Transformer：query-key-value 交互、mask、位置信息、归一化、前馈模块以及高效自回归推理。